### P1 — Regression

In [1]:
from mibici.modeling.p1 import load_p1_data

daily_demand = load_p1_data()

analysis_data = daily_demand.copy()
analysis_data["day_of_week"] = analysis_data["date"].dt.dayofweek

2026-10-18 09:00:07.417 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: X:\Repositorios\Personal\Kaggle\notebooks\mibici\mibici_predictions


2026-10-18 09:00:17.357 | INFO     | mibici.dataset:load_raw:166 - Partición train: 2024-10-01 a 2026-02-22
2026-10-18 09:00:17.358 | INFO     | mibici.dataset:load_raw:170 - Viajes cargados: 6,311,111


In [2]:
station_weekday_profile = analysis_data.pivot_table(
    index="station_id",
    columns="day_of_week",
    values="departures",
    aggfunc="mean",
).reindex(columns=range(7))

station_weekday_profile.columns = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]

display(station_weekday_profile.round(2).head(10))

,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
station_id,,,,,,,
2,56.86,61.85,63.23,62.00,61.56,40.01,25.44
3,54.29,60.48,58.92,57.19,55.00,40.78,28.79
4,66.32,69.67,70.58,67.25,63.32,43.41,51.68
5,52.94,59.58,60.90,58.52,55.12,37.93,27.73
6,27.58,27.68,27.23,26.92,26.48,20.89,16.44
8,28.50,29.79,30.55,30.82,30.52,24.74,19.84
9,41.74,43.00,42.16,43.58,43.18,31.70,21.85
10,42.65,43.10,45.21,43.34,41.63,30.34,23.10
11,198.19,212.36,214.29,208.41,200.36,142.34,106.90


En las estaciones mostradas, el promedio de salidas cambia según
el día de la semana. La magnitud y el patrón difieren por estación.
Se propone evaluar estación y día de la semana como predictores.

In [3]:
analysis_data = analysis_data.sort_values(
    ["station_id", "date"]
).copy()

analysis_data["departures_lag_1"] = (
    analysis_data.groupby("station_id")["departures"].shift(1)
)

display(
    analysis_data[
        ["station_id", "date", "departures", "departures_lag_1"]
    ].head(10)
)

,station_id,date,departures,departures_lag_1
0,2,2024-10-01,21,NaN
1,2,2024-10-02,74,21.0
2,2,2024-10-03,83,74.0
3,2,2024-10-04,63,83.0
4,2,2024-10-05,44,63.0
5,2,2024-10-06,30,44.0
6,2,2024-10-07,76,30.0
7,2,2024-10-08,83,76.0
8,2,2024-10-09,86,83.0
9,2,2024-10-10,75,86.0


In [4]:
lag_pairs = analysis_data[
    ["departures", "departures_lag_1"]
].dropna()

display(lag_pairs.corr())

,departures,departures_lag_1
departures,1.000000,0.892547
departures_lag_1,0.892547,1.000000


In [5]:
station_lag_correlation = (
    analysis_data.groupby("station_id")[
        ["departures", "departures_lag_1"]
    ]
    .corr()
    .loc[(slice(None), "departures"), "departures_lag_1"]
    .droplevel(1)
    .rename("lag_1_correlation")
)

display(station_lag_correlation.describe())

count    371.000000
mean       0.329379
std        0.131413
min       -0.196933
25%        0.268313
50%        0.348616
75%        0.402697
max        0.775342
Name: lag_1_correlation, dtype: float64

Alcance: partición `train` (2024-10-01 a 2026-02-22), 371 estaciones,
calendario exploratorio de `load_p1_data`.

El rezago de un día presenta correlación global alta (0.893),
pero una mediana por estación de 0.349. Se conserva como variable
candidata; su utilidad se evaluará mediante validación temporal.

In [6]:
analysis_data["departures_lag_7"] = (
    analysis_data.groupby("station_id")["departures"].shift(7)
)

In [7]:
comparison_data = analysis_data.dropna(
    subset=["departures", "departures_lag_1", "departures_lag_7"]
)

display(
    comparison_data[
        ["departures", "departures_lag_1", "departures_lag_7"]
    ].corr()
)

,departures,departures_lag_1,departures_lag_7
departures,1.000000,0.893303,0.933314
departures_lag_1,0.893303,1.000000,0.878426
departures_lag_7,0.933314,0.878426,1.000000


In [8]:
station_correlations = (
    comparison_data.groupby("station_id")[
        ["departures", "departures_lag_1", "departures_lag_7"]
    ]
    .corr()
    .xs("departures", level=1)[
        ["departures_lag_1", "departures_lag_7"]
    ]
)

display(station_correlations.describe())

,departures_lag_1,departures_lag_7
count,371.000000,371.000000
mean,0.332518,0.513251
std,0.132368,0.159783
min,-0.199418,0.134390
25%,0.270181,0.392252
50%,0.351867,0.545845
75%,0.405800,0.635939
max,0.777316,0.868734


In [9]:
analysis_data["departures_mean_7"] = (
    analysis_data.groupby("station_id")["departures"]
    .transform(
        lambda values: values.shift(1).rolling(
            window=7,
            min_periods=7,
        ).mean()
    )
)

In [10]:
display(
    analysis_data[
        [
            "station_id", "date", "departures",
            "departures_lag_1", "departures_lag_7",
            "departures_mean_7",
        ]
    ].head(12)
)

,station_id,date,departures,departures_lag_1,departures_lag_7,departures_mean_7
0,2,2024-10-01,21,NaN,NaN,NaN
1,2,2024-10-02,74,21.0,NaN,NaN
2,2,2024-10-03,83,74.0,NaN,NaN
3,2,2024-10-04,63,83.0,NaN,NaN
4,2,2024-10-05,44,63.0,NaN,NaN
5,2,2024-10-06,30,44.0,NaN,NaN
6,2,2024-10-07,76,30.0,NaN,NaN
7,2,2024-10-08,83,76.0,21.0,55.857143
8,2,2024-10-09,86,83.0,74.0,64.714286
9,2,2024-10-10,75,86.0,83.0,66.428571


In [11]:
feature_columns = [
    "departures_lag_1",
    "departures_lag_7",
    "departures_mean_7",
]

comparison_data = analysis_data.dropna(
    subset=["departures"] + feature_columns
)

station_correlations = (
    comparison_data.groupby("station_id")[
        ["departures"] + feature_columns
    ]
    .corr()
    .xs("departures", level=1)[feature_columns]
)

display(station_correlations.describe())

,departures_lag_1,departures_lag_7,departures_mean_7
count,371.000000,371.000000,371.000000
mean,0.332518,0.513251,0.318809
std,0.132368,0.159783,0.102917
min,-0.199418,0.134390,0.055732
25%,0.270181,0.392252,0.252688
50%,0.351867,0.545845,0.310967
75%,0.405800,0.635939,0.370963
max,0.777316,0.868734,0.786511


#### Conclusiones para P1

Se proponen como variables candidatas: estación, día de la semana,
salidas del día anterior, salidas de hace siete días y promedio
de los siete días anteriores.

El rezago semanal presenta la mayor correlación mediana por estación.
Se utilizará también como referencia de predicción.

Los faltantes iniciales de los rezagos representan historia
no disponible y no se rellenarán automáticamente con cero.

La utilidad de las variables se comprobará mediante validación
temporal; la correlación no determina su selección final.

### P2 — Classification